# Проект для «Викишоп»

Интернет-магазин «Викишоп» запускает новый сервис. Теперь пользователи могут редактировать и дополнять описания товаров, как в вики-сообществах. То есть клиенты предлагают свои правки и комментируют изменения других. Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию. 

Обучите модель классифицировать комментарии на позитивные и негативные. В вашем распоряжении набор данных с разметкой о токсичности правок.

Постройте модель со значением метрики качества *F1* не меньше 0.75. 

**Инструкция по выполнению проекта**

1. Загрузите и подготовьте данные.
2. Обучите разные модели. 
3. Сделайте выводы.

Для выполнения проекта применять *BERT* необязательно, но вы можете попробовать.

**Описание данных**

Данные находятся в файле `toxic_comments.csv`. Столбец *text* в нём содержит текст комментария, а *toxic* — целевой признак.

## Подготовка

Для начала загрузим необходимые нам библиотеки

In [1]:
!pip install spacy -q
!pip install spacy-lookups-data -q
!pip install optuna -q
!pip install optuna-integration -q
!pip install tqdm -q

In [2]:
from collections import Counter
import numpy as np
import pandas as pd
import scipy
from scipy.sparse import vstack
from tqdm.notebook import tqdm
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, ElasticNet
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from catboost import CatBoostClassifier, Pool
import lightgbm as LGBMClassifier
import lightgbm as lgb
from sklearn.metrics import f1_score
import sys
from transformers import AutoTokenizer, AutoModelForMaskedLM
import gc
from sklearn.utils import shuffle
import spacy
import re
import optuna
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 20)
pd.set_option('max_colwidth', 1000)
RANDOM_STATE = 42
tqdm.pandas()

[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Теперь загрузим и посмотрим на наш датасет

In [3]:
df = pd.read_csv("/datasets/toxic_comments.csv")
df.head(2)

,Unnamed: 0,text,toxic
0,0,"Explanation\nWhy the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27",0
1,1,"D'aww! He matches this background colour I'm seemingly stuck with. Thanks. (talk) 21:51, January 11, 2016 (UTC)",0


Выведем всю предварительную информацию чтоб получше изучить данные

In [4]:
print(df.info(), '\n', df['toxic'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159292 entries, 0 to 159291
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  159292 non-null  int64 
 1   text        159292 non-null  object
 2   toxic       159292 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.6+ MB
None 
 0    143106
1     16186
Name: toxic, dtype: int64


Мы можем констатировать факты о том, что этот датасет имеет ярко-выроженный дисбаланс классов, это непременно нужно взять во внимание, ведь это отразится на качестве метрик, нам непременно нужно проверить на тот факт, что данные могут иметь дубликаты

In [5]:
row = df.duplicated()
war = df['text'].duplicated()
print(row.sum(), war.sum())

0 0


Отлично! теперь мы знаем что нет пропусков и дубликатов, это превосходная новость, в дальнейшем из-за ограниченных вычислительных ресурсов и увиличении скорости выполнения кода, мы вынужденны ограничить выборку до 100_000 строчек

In [6]:

toxic = df.sample(100000, random_state=RANDOM_STATE)
print(toxic.info(), '\n', toxic['toxic'].value_counts())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 100000 entries, 31015 to 110479
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  100000 non-null  int64 
 1   text        100000 non-null  object
 2   toxic       100000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.1+ MB
None 
 0    89865
1    10135
Name: toxic, dtype: int64


Хорошо, первое частичное преобразование выполненно

Видим, что нам необходимо почистить коментарии от символов, перевести в нижний регистр, а слова вернуть в лемматической форме

In [7]:
disabled_pipes = [ "parser",  "ner"]
nlp = spacy.load('en_core_web_sm', disable=disabled_pipes)
def clear_text(text):
    return " ".join(re.sub(r'[^a-zA-Z]', ' ', text.lower()).split())
def lemmatize(text):
    global nlp
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])
toxic['clear_text'] = toxic['text'].apply(clear_text)
toxic['lemmatized_text'] = toxic['clear_text'].progress_apply(lemmatize)

  0%|          | 0/100000 [00:00<?, ?it/s]

In [8]:
toxic.head(5)

,Unnamed: 0,text,toxic,clear_text,lemmatized_text
31015,31055,"Sometime back, I just happened to log on to www.izoom.in with a friend’s reference and I was amazed to see the concept Fresh Ideas Entertainment has come up with. So many deals… all under one roof. This website is very user friendly and easy to use and is fun to be on.\nYou have Gossip, Games, Facts… Another exciting feature to add to it is Face of the Week… Every week, 4 new faces are selected and put up as izoom faces. It’s great to have been selected in four out of a group of millions. \nThis new start up has already got many a deals in its kitty. Few of them being TheFortune Hotel, The Beach… are my personal favorites. izoom.in has a USP of mobile coupons. Coupons are available even when a user cannot access internet. You just need to SMS izoom support to 56767 and you get attended immediately.\nAll I can say is izoom.in is a must visit website for everyone before they go out for shopping or dining or for outing.\nCheers!!!",0,sometime back i just happened to log on to www izoom in with a friend s reference and i was amazed to see the concept fresh ideas entertainment has come up with so many deals all under one roof this website is very user friendly and easy to use and is fun to be on you have gossip games facts another exciting feature to add to it is face of the week every week new faces are selected and put up as izoom faces it s great to have been selected in four out of a group of millions this new start up has already got many a deals in its kitty few of them being thefortune hotel the beach are my personal favorites izoom in has a usp of mobile coupons coupons are available even when a user cannot access internet you just need to sms izoom support to and you get attended immediately all i can say is izoom in is a must visit website for everyone before they go out for shopping or dining or for outing cheers,sometime back I just happen to log on to www izoom in with a friend s reference and I be amazed to see the concept fresh idea entertainment have come up with so many deal all under one roof this website be very user friendly and easy to use and be fun to be on you have gossip game fact another exciting feature to add to it be face of the week every week new face be select and put up as izoom face it s great to have be select in four out of a group of million this new start up have already get many a deal in its kitty few of they be thefortune hotel the beach be my personal favorite izoom in have a usp of mobile coupon coupon be available even when a user can not access internet you just need to sms izoom support to and you get attend you all I can say be izoom in be a must visit website for everyone before they go out for shopping or dining or for outing cheer
102832,102929,"""\n\nThe latest edit is much better, don't make this article state """"super."""" at all. 71.237.70.49 """,0,the latest edit is much better don t make this article state super at all,the late edit be much well don t make this article state super at all
67317,67385,""" October 2007 (UTC)\n\nI would think you'd be able to get your point across, and be immune to any objections, were you to simply embellish the second sentence of the article by changing """"he was schooled at Thornleigh Salesian College"""" to """"he was schooled at (the then all-Catholic) Thornleigh Salesian College"""". \n\nGood suggestion from an Anon - what do you think? Rgds, - 07:53, 5""",0,october utc i would think you d be able to get your point across and be immune to any objections were you to simply embellish the second sentence of the article by changing he was schooled at thornleigh salesian college to he was schooled at the then all catholic thornleigh salesian college good suggestion from an anon what do you think rgds,october utc I would think you d be able to get your point across and be immune to any objection be you to simply embellish the second sentence of the article by change he be school at t

Видим, что преобразование прошло успешно, теперь нам необходимо векторизовать наши комментарии, попутно убирая стоп слова, чтобы улучшить как скорость обучения так и уменьшить своеборазный шум в данных

In [9]:
features = toxic['lemmatized_text']
target = toxic['toxic']
features_train_val, features_test, target_train_val, target_test = train_test_split(
    features,
    target, 
    test_size=0.1, 
    random_state=12345, 
    stratify=target)

features_train, features_val, target_train, target_val = train_test_split(
    features_train_val,
    target_train_val, 
    test_size=0.10, 
    random_state=12345, 
    stratify=target_train_val)

#векторизация
vect = TfidfVectorizer()
X_train = vect.fit_transform(features_train)
X_val = vect.transform(features_val)
X_test = vect.transform(features_test)

# Целевые переменные
y_train = target_train.reset_index(drop=True)
y_val = target_val.reset_index(drop=True)
y_test = target_test.reset_index(drop=True)

## Обучение

Теперь обучим на наших подготовленных выборках несколько моделей

In [10]:
lgbm_tfidf = lgb.LGBMClassifier(random_state=12345, learning_rate=0.1, n_estimators=200, n_jobs=-1)
lgbm_tfidf.fit(X_train, y_train)
lgbm_tfidf_pred = lgbm_tfidf.predict(X_val)
print(f1_score(y_val, lgbm_tfidf_pred))

model_tfidf = LogisticRegression(class_weight='balanced', random_state=12345, n_jobs=-1)
model_tfidf.fit(X_train, y_train)
model_tfidf_pred = model_tfidf.predict(X_val)
print(f1_score(y_val, model_tfidf_pred))


model2_tfidf = CatBoostClassifier(silent=True, thread_count=-1)
model2_tfidf.fit(X_train, y_train)
model2_tfidf_pred = model2_tfidf.predict(X_val)
print(f1_score(y_val, model2_tfidf_pred))

0.7733168622606547


/opt/conda/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:763: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.7317770366841353
0.7691334598355472


0.77! Именно с таким результатом побеждает LGBMClassifier! Но, для того чтобы окончательно выбрать победителя а так же улучшить score нам нужно подобрать гиперпараметры для моделей а затем опять сравнить моделей, однако, дефолтные модели показывают результаты которые соответствуют уровню запрашиваемомоу от заказчика, чтож давайте освободим наши застоявшиеся вычислительные ресурсы и выполним проверку на тестовой выборке

In [11]:
del df, nlp, toxic, features, target, vect, lgbm_tfidf_pred, model_tfidf, model_tfidf_pred, model2_tfidf, model2_tfidf_pred

LogisticRegression

Лучшая f1_score: 0.73

LGBMClassifier

Лучшая f1_score: 0.77

CatBoostClassifier

Лучшая f1_score: 0.76


Самая лучшая модель - LGBMClassifier

После выбора модели нам нужно проверить её на тестовой выборке которую мы опредилили в самом начале, чтобы она ни где не участвовала и была показательна

In [15]:
X_full_train = vstack([X_train, X_val])
y_full_train = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)
#model = LogisticRegression(C=13, solver='saga', penalty='l1', random_state=42, max_iter=1000)

lgbm_tfidf.fit(X_full_train, y_full_train)
preds = lgbm_tfidf.predict(X_test)
f1 = f1_score(y_test, preds)
f1

0.7645051194539249

In [ ]:
q

## Выводы

Наконец мы обучили различные модели классифицировать комментарии на позитивные и негативные со значением метрики качества F1 не меньше 0.75, ведь это непосредственно требование заказчика.

Для этого нам было необходимо подготовить данные и обработать их в формат который бы могли использовать наши модели. После чего мы обучили различные модели , Далее был этап сравнения, на котором мы выяснили, что самая лучшая модель (на сколько мы можем это выяснить) это LGBMClassifier с лучшим f1_score: 0.76